# 34. 합성 평가셋 — Stage 1 조건 구조화

사전 등록: `docs/plans/NLR_EVALSET_PREREGISTRATION.md`
입력: 노트북 33 이 고정한 `evaluation_data/nlr_evalset/generated_{A,B}.csv` (600문장)

`spec.md` §3 ②-a 대로 **LLM 이 문장을 조건 JSON 으로 구조화한다.**
채점하지 않는다. 기준선 측정은 노트북 35 에서 한다.

## 0. 실행 조건과 한계

- **API 를 호출한다.** GMS `gpt-5.4-nano`. 600문장 × 1회
- **프롬프트를 새로 쓰지 않는다.** `15_stage1_llm_prompt_v1.txt` 를 그대로 쓴다.
  Golden Set 200 이 이 프롬프트로 만들어졌으므로 바꾸면 비교가 깨진다
- **체크포인트에 원본 응답을 저장한다.** 재채점할 때 API 를 다시 부르지 않는다
  (`WORKING_NOTES.md` §3 패턴). 이미 처리된 문장은 건너뛴다
- **재시도는 네트워크 오류와 스키마 검증 실패에만.** `spec.md` §5.1 —
  결과가 마음에 안 든다는 이유로 재호출하지 않는다
- 키가 만료되면 `HTTP 401` 이다. **재시도하지 말고 멈춘다** (`WORKING_NOTES.md` §1)
- 평가 데이터와 사전 파일은 읽기만 한다

In [1]:
import hashlib
import json
import os
import pathlib
import time
import urllib.error
import urllib.request

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False
# True면 체크포인트를 무시하고 전부 다시 호출한다. 평소에는 False.
FORCE_RERUN = False

GMS_URL = "https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions"
MODEL = "gpt-5.4-nano"
TIMEOUT_SEC = 60
MAX_ATTEMPTS = 3          # 네트워크 오류·스키마 실패에만 쓴다
PROMPT_VERSION = "15_stage1_llm_prompt_v1"

SCHEMA_KEYS = ["scent_preference", "context", "performance",
               "avoid", "additional_requirements"]

print(f"REPORT_ONLY={REPORT_ONLY} / FORCE_RERUN={FORCE_RERUN} / model={MODEL}")

REPORT_ONLY=False / FORCE_RERUN=False / model=gpt-5.4-nano


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
EVAL_DIR = PROJECT_ROOT / "evaluation_data" / "nlr_evalset"

INPUT_PATHS = {
    "generated_a": EVAL_DIR / "generated_A.csv",
    "generated_b": EVAL_DIR / "generated_B.csv",
    "checksums": EVAL_DIR / "checksums.json",
    "prompt": OUTPUT_DIR / "15_stage1_llm_prompt_v1.txt",
    "answer_key": OUTPUT_DIR / "32_evalset_answer_key.csv",
}
OUTPUT_PATHS = {
    "checkpoint": OUTPUT_DIR / "34_evalset_stage1_checkpoint.csv",
    "summary": OUTPUT_DIR / "34_evalset_stage1_summary.csv",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {p.resolve() for p in INPUT_PATHS.values()} | {
    (PROJECT_ROOT / "perfumes.csv").resolve(),
    (PROJECT_ROOT / "perfumes.jsonl").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (OUTPUT_DIR / "15_llm_stage1_checkpoint.csv").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
generated_a,1ac78a370fa27581
generated_b,7c3c869594c43a2f
checksums,78385413b642084b
prompt,24f9c2c8f833c134
answer_key,d81e0f4fb2f1e1cd


## 2. 사전 등록 — 호출하기 전에 고정한다

프롬프트 해시를 박아두면 나중에 "무엇으로 돌렸는가" 를 증명할 수 있다.

In [3]:
PROMPT_TEXT = INPUT_PATHS["prompt"].read_text(encoding="utf-8")
PROMPT_SHA = hashlib.sha256(PROMPT_TEXT.encode("utf-8")).hexdigest()

PREREG = {
    "목적": "합성 평가셋 600문장을 spec §3 ②-a 의 조건 JSON 으로 구조화",
    "프롬프트": f"{PROMPT_VERSION} (수정 없음) sha256={PROMPT_SHA[:16]}…",
    "모델": MODEL,
    "호출 수": "문장당 1회. 체크포인트가 있으면 건너뛴다",
    "재시도": f"네트워크 오류·JSON 파싱 실패·스키마 위반에만, 최대 {MAX_ATTEMPTS}회",
    "재시도 금지": "결과가 마음에 안 든다는 이유로 재호출하지 않는다 (spec §5.1)",
    "401 처리": "재시도하지 않고 중단한다 (WORKING_NOTES §1)",
    "이 노트북이 안 하는 것": "채점. 기준선은 노트북 35",
}
display(pd.Series(PREREG, name="사전 등록").to_frame())
print(f"\n프롬프트 {len(PROMPT_TEXT)}자 / sha256 {PROMPT_SHA}")

,사전 등록
목적,합성 평가셋 600문장을 spec §3 ②-a 의 조건 JSON 으로 구조화
프롬프트,15_stage1_llm_prompt_v1 (수정 없음) sha256=4f0a368d98b41338…
모델,gpt-5.4-nano
호출 수,문장당 1회. 체크포인트가 있으면 건너뛴다
재시도,"네트워크 오류·JSON 파싱 실패·스키마 위반에만, 최대 3회"
재시도 금지,결과가 마음에 안 든다는 이유로 재호출하지 않는다 (spec §5.1)
401 처리,재시도하지 않고 중단한다 (WORKING_NOTES §1)
이 노트북이 안 하는 것,채점. 기준선은 노트북 35



프롬프트 1401자 / sha256 4f0a368d98b413384935d4bb54d36d19b60895904d16733d526a1a82737cc493


## 3. 입력 — 600문장

In [4]:
frames = []
for arm in ("A", "B"):
    d = pd.read_csv(INPUT_PATHS[f"generated_{arm.lower()}"])
    d["arm"] = arm
    d["query_id"] = arm + "-" + d["perfume_id"].astype(str) + "-" + d["sentence_no"].astype(str)
    frames.append(d)
queries = pd.concat(frames, ignore_index=True)[
    ["query_id", "arm", "perfume_id", "sentence_no", "sentence", "batch"]]

if queries["query_id"].duplicated().any():
    raise RuntimeError("query_id 가 중복된다")
print(f"문장 {len(queries)}건 / 갈래 {queries['arm'].nunique()}개 / 향수 {queries['perfume_id'].nunique()}개")
display(queries.head(3))

문장 600건 / 갈래 2개 / 향수 100개


,query_id,arm,perfume_id,sentence_no,sentence,batch
0,A-6-1,A,6,1,하얀 꽃 향이 진하면서 깨끗한 비누처럼 밝은 느낌이 있었으면 좋겠어. 나무 향과 보송한 분냄새도 은근...,1
1,A-6-2,A,6,2,단정하고 클래식한 향을 찾고 있어. 꽃 향과 깨끗한 세탁물 같은 느낌이 나고 포근한 분 향이 남았으면...,1
2,A-6-3,A,6,3,꽃다발 같은 향에 숲의 나무 냄새가 섞였으면 좋겠어. 상쾌하면서도 화장품 파우더처럼 보송한 느낌도 있...,1


## 4. 호출

이미 체크포인트에 있는 문장은 건너뛴다. 중간에 끊겨도 다시 실행하면 이어서 한다.

In [5]:
def load_key():
    """.env 에서 GMS_KEY 를 읽는다. 값을 출력하지 않는다. str."""
    key = os.environ.get("GMS_KEY", "")
    if not key:
        env = PROJECT_ROOT / ".env"
        if env.is_file():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("GMS_KEY"):
                    key = line.partition("=")[2].strip()
    if not key:
        raise RuntimeError("GMS_KEY 가 없습니다. .env 를 확인하세요.")
    return key


def extract_json(text):
    """응답에서 JSON object 를 꺼낸다. dict 또는 None."""
    t = str(text).strip()
    if t.startswith("```"):
        t = t.split("```")[1] if "```" in t[3:] else t[3:]
        t = t[4:] if t.lower().startswith("json") else t
    i, j = t.find("{"), t.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(t[i:j + 1])
    except ValueError:
        return None


def schema_ok(obj):
    """spec §3 ②-a 스키마를 만족하는가. (bool, str)."""
    if not isinstance(obj, dict):
        return False, "dict 아님"
    miss = [k for k in SCHEMA_KEYS if k not in obj]
    if miss:
        return False, f"key 누락 {miss}"
    for k in ("scent_preference", "avoid", "additional_requirements"):
        if not isinstance(obj[k], list):
            return False, f"{k} 가 list 아님"
    if not isinstance(obj.get("context"), dict) or not isinstance(obj.get("performance"), dict):
        return False, "context/performance 가 dict 아님"
    return True, ""


def call_once(sentence, key):
    """LLM 1회 호출. (raw_text, usage, error). error 는 없으면 None."""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "system", "content": PROMPT_TEXT},
                     {"role": "user", "content": str(sentence)}],
    }).encode("utf-8")
    req = urllib.request.Request(GMS_URL, data=body, headers={
        "Authorization": f"Bearer {key}", "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=TIMEOUT_SEC) as r:
            o = json.loads(r.read())
        return o["choices"][0]["message"]["content"], o.get("usage", {}), None
    except urllib.error.HTTPError as e:
        detail = ""
        try:
            detail = e.read().decode()[:200]
        except Exception:
            pass
        return None, {}, f"HTTP {e.code} {detail}"
    except Exception as e:
        return None, {}, f"{type(e).__name__}: {str(e)[:160]}"

In [6]:
CKPT = OUTPUT_PATHS["checkpoint"]
done = {}
if CKPT.is_file() and not FORCE_RERUN:
    prev = pd.read_csv(CKPT)
    done = {r["query_id"]: r for r in prev.to_dict("records")}
    print(f"체크포인트에서 {len(done)}건을 이어받는다")
else:
    print("체크포인트 없음 — 처음부터 호출한다" if not FORCE_RERUN else "FORCE_RERUN — 전부 다시 호출")

todo = [r for r in queries.to_dict("records") if r["query_id"] not in done]
print(f"호출 대상 {len(todo)}건 (전체 {len(queries)}건)")

if todo and not REPORT_ONLY:
    key = load_key()
    records, t_start = [], time.time()
    for n, row in enumerate(todo, 1):
        raw, usage, err, attempts, status = None, {}, None, 0, "ok"
        parsed, why = None, ""
        for attempt in range(1, MAX_ATTEMPTS + 1):
            attempts = attempt
            t0 = time.time()
            raw, usage, err = call_once(row["sentence"], key)
            if err and err.startswith("HTTP 401"):
                raise RuntimeError(f"GMS 키 만료로 보인다 — 재시도하지 않고 멈춘다. {err}")
            if err:
                status = "api_error"
                if attempt < MAX_ATTEMPTS:
                    time.sleep(2 * attempt)
                    continue
                break
            parsed = extract_json(raw)
            if parsed is None:
                status, why = "parse_fail", "JSON 을 찾지 못함"
                if attempt < MAX_ATTEMPTS:
                    continue
                break
            ok, why = schema_ok(parsed)
            if ok:
                status = "ok"
                break
            status = "schema_fail"
            if attempt >= MAX_ATTEMPTS:
                break
        records.append({
            **{k: row[k] for k in ("query_id", "arm", "perfume_id", "sentence_no",
                                   "sentence", "batch")},
            "raw_response": raw, "parsed": json.dumps(parsed, ensure_ascii=False)
            if parsed is not None else "",
            "status": status, "error": err or why, "attempts": attempts,
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "latency_seconds": round(time.time() - t0, 3),
            "model_name": MODEL, "prompt_version": PROMPT_VERSION,
            "prompt_sha256": PROMPT_SHA,
            "run_timestamp": pd.Timestamp.now("UTC").isoformat(),
        })
        if n % 50 == 0 or n == len(todo):
            el = time.time() - t_start
            print(f"  {n}/{len(todo)}  경과 {el/60:.1f}분  "
                  f"남은 예상 {el/n*(len(todo)-n)/60:.1f}분", flush=True)
    new = pd.DataFrame(records)
else:
    new = pd.DataFrame()
    if REPORT_ONLY:
        print("[REPORT_ONLY] 호출 생략")

ckpt = pd.concat([pd.DataFrame(list(done.values())), new], ignore_index=True) \
    if done or len(new) else pd.DataFrame()
print(f"체크포인트 총 {len(ckpt)}건")

체크포인트 없음 — 처음부터 호출한다
호출 대상 600건 (전체 600건)


  50/600  경과 1.5분  남은 예상 16.3분


  100/600  경과 2.8분  남은 예상 14.0분


  150/600  경과 4.1분  남은 예상 12.2분


  200/600  경과 5.3분  남은 예상 10.6분


...

  450/600  경과 11.5분  남은 예상 3.8분


  500/600  경과 12.7분  남은 예상 2.5분


  550/600  경과 13.9분  남은 예상 1.3분


  600/600  경과 15.2분  남은 예상 0.0분


체크포인트 총 600건


## 5. 검증

In [7]:
if len(ckpt):
    summary = (ckpt.groupby(["arm", "status"]).size()
               .unstack(fill_value=0))
    display(summary)
    ok_rate = (ckpt["status"] == "ok").mean()
    print(f"스키마 통과율 {ok_rate:.1%}  ({(ckpt['status']=='ok').sum()}/{len(ckpt)})")
    print(f"평균 지연 {ckpt['latency_seconds'].mean():.2f}초 / "
          f"p95 {ckpt['latency_seconds'].quantile(0.95):.2f}초")
    tok = ckpt[["prompt_tokens", "completion_tokens"]].sum()
    print(f"토큰 합계 prompt {tok['prompt_tokens']:,.0f} / completion {tok['completion_tokens']:,.0f}")
    bad = ckpt[ckpt["status"] != "ok"]
    if len(bad):
        print(f"\n실패 {len(bad)}건 — 사유 상위")
        display(bad["error"].value_counts().head(5).to_frame("건수"))
else:
    print("체크포인트가 비어 있다")

status,ok
arm,
A,300
B,300


스키마 통과율 100.0%  (600/600)
평균 지연 1.52초 / p95 2.01초
토큰 합계 prompt 383,790 / completion 66,064


### 뽑힌 조건이 얼마나 되나

여기서 **조건이 0개인 문장** 이 나오면 그 문장은 검색 자체를 못 한다.
`spec.md` §5.3 이 *"향과 무관한 입력은 조건이 하나도 안 잡히므로 감지된다"* 고 한 지점이다.

In [8]:
if len(ckpt):
    def field_counts(js):
        """구조화 결과의 필드별 개수. dict."""
        try:
            o = json.loads(js) if js else {}
        except ValueError:
            o = {}
        ctx = o.get("context") or {}
        perf = o.get("performance") or {}
        return {
            "scent": len(o.get("scent_preference") or []),
            "avoid": len(o.get("avoid") or []),
            "additional": len(o.get("additional_requirements") or []),
            "season": len(ctx.get("season") or []),
            "daypart": len(ctx.get("daypart") or []),
            "gender": len(ctx.get("gender") or []),
            "intensity": 1 if perf.get("intensity") else 0,
            "longevity": 1 if perf.get("longevity") else 0,
        }

    fc = pd.DataFrame([field_counts(j) for j in ckpt["parsed"]], index=ckpt.index)
    fc["arm"] = ckpt["arm"]
    fc["조건 합계"] = fc.drop(columns=["arm"]).sum(axis=1)
    display(fc.groupby("arm").mean(numeric_only=True).round(2))
    print("\n조건이 하나도 없는 문장")
    display((fc["조건 합계"] == 0).groupby(fc["arm"]).agg(["sum", "mean"]))

,scent,avoid,additional,season,daypart,gender,intensity,longevity,조건 합계
arm,,,,,,,,,
A,2.54,0.06,3.08,0.10,0.00,0.00,0.14,0.01,5.94
B,1.37,0.13,3.38,0.38,0.15,0.14,0.08,0.00,5.64



조건이 하나도 없는 문장


,sum,mean
arm,,
A,0,0.0
B,0,0.0


## 6. 저장

In [9]:
if len(ckpt):
    write_output(OUTPUT_PATHS["checkpoint"],
                 lambda p: ckpt.to_csv(p, index=False, encoding="utf-8-sig"))
    out = ckpt.groupby(["arm", "status"]).size().reset_index(name="건수")
    write_output(OUTPUT_PATHS["summary"],
                 lambda p: out.to_csv(p, index=False, encoding="utf-8-sig"))

저장: analysis_outputs\34_evalset_stage1_checkpoint.csv
저장: analysis_outputs\34_evalset_stage1_summary.csv


## 7. 가드 검증

In [10]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("다음 — 노트북 35 에서 기준선을 측정한다.")
print("  구조화 결과 → 정규화 → 사전 조회 → 검색(하드 AND / 소프트 점수) → 채점")

입력 해시 불변 확인: generated_a, generated_b, checksums, prompt, answer_key

다음 — 노트북 35 에서 기준선을 측정한다.
  구조화 결과 → 정규화 → 사전 조회 → 검색(하드 AND / 소프트 점수) → 채점
